# Week 4 studio — REFERENCE SOLUTION (instructor-only)

**Task brief:** [`README.md`](README.md) · **Lesson plan:** [`../../weeks/week-04.md`](../../weeks/week-04.md) · **Given engine:** [`gridworld.py`](gridworld.py) · **Duel helpers:** [`duel.py`](duel.py)

This notebook is the **teaching walkthrough** of the week-4 studio. It **imports the
reference code from [`solution.py`](solution.py)** — it never re-pastes it — so the
answer shown here is byte-for-byte the one the provided test grades. The correctness
guarantee is separate: `python3 studios/_verify_solutions.py week-04` runs the
unmodified `test_heuristics.py` against `solution.py`.

> Do not distribute. Excluded from students via `studios/.gitignore`.

In [ ]:
# --- bootstrap: put the week folder (for solution/gridworld/duel) and the repo
# root (for aicourse) on the path, so this runs from anywhere. ---
import sys, pathlib
here = pathlib.Path.cwd()
week = here if (here / "solution.py").exists() else here / "studios" / "week-04"
root = week.parent.parent
for p in (str(week), str(root)):
    if p not in sys.path:
        sys.path.insert(0, p)

import inspect
import solution
from gridworld import GridProblem, astar, load_instances, render, MIN_COST, MAX_COST

BANK = load_instances()
print(f"MIN_COST={MIN_COST} MAX_COST={MAX_COST}; sizes:", sorted(BANK))

## Task 1 — four heuristics (three admissible, one deliberately bad)

`_optimal_cost` in the test uses UCS (A* with `h=0`) as ground truth — it is optimal
because all step costs are > 0. Every claim below is measured against it.

In [ ]:
for fn in (solution.h_zero, solution.h_manhattan, solution.h_manhattan_scaled,
           solution.h_euclidean_scaled, solution.h_bad):
    print(inspect.getsource(fn))

def optimal_cost(grid):
    node, _ = astar(GridProblem(grid), lambda p, st: 0)
    return node.g

### The guarantees that must HOLD

- `test_h_zero_admissible_and_optimal` — `h_zero` is admissible by definition ⇒ A* returns the true optimum.
- `test_manhattan_scaled_is_admissible` — `Manhattan × MIN_COST` never overestimates the true remaining cost.
- `test_manhattan_scaled_dominates_zero` — a dominating heuristic expands **no more** nodes than `h_zero`.
- `test_manhattan_scaled_returns_optimal_cost` — and it still returns the optimum.

In [ ]:
# h_zero optimal on all 40 grids
for size in BANK:
    for grid in BANK[size]:
        node, _ = astar(GridProblem(grid), solution.h_zero)
        assert node.g == optimal_cost(grid)
print("h_zero admissible => optimal on all 40 grids")

# manhattan_scaled: admissible (sampled), dominates zero, still optimal
for size in BANK:
    for grid in BANK[size]:
        prob = GridProblem(grid)
        for state in [prob.initial, prob.goal, (prob.rows // 2, prob.cols // 2)]:
            sub = GridProblem(grid); sub.initial = state
            rem, _ = astar(sub, lambda p, st: 0)
            assert solution.h_manhattan_scaled(prob, state) <= rem.g + 1e-9
        _, exp_zero = astar(prob, solution.h_zero)
        _, exp_manh = astar(prob, solution.h_manhattan_scaled)
        assert exp_manh <= exp_zero
        node, _ = astar(prob, solution.h_manhattan_scaled)
        assert node.g == optimal_cost(grid)
print("h_manhattan_scaled: admissible, dominates h_zero, and stays optimal on all 40 grids")

## The guarantee fails — an inadmissible heuristic breaks optimality

This is **the point of the week** (`test_inadmissible_heuristic_returns_suboptimal_path`).
`h_bad = MAX_COST × Manhattan` assumes every remaining step costs 8, so it overestimates
wherever the optimal path runs through cheap `.` cells. A* is optimal *because* of the
admissibility property; `h_bad` shows that property fail on request — A* returns a path
strictly costlier than the true optimum on at least one instance.

In [ ]:
broken = 0; total = 0; example = None
for size in BANK:
    for grid in BANK[size]:
        node, _ = astar(GridProblem(grid), solution.h_bad)
        opt = optimal_cost(grid)
        if node.g > opt:
            broken += 1
            if example is None:
                example = (size, node.g, opt)
        total += 1
assert broken > 0
sz, got, opt = example
print(f"GUARANTEE FAIL confirmed: h_bad broke optimality on {broken}/{total} instances.")
print(f"  e.g. a {sz}x{sz} grid: A*+h_bad returned cost {got}, true optimum is {opt}.")
# A* with an admissible heuristic on the SAME bank never breaks — the contrast:
assert all(astar(GridProblem(g), solution.h_manhattan_scaled)[0].g == optimal_cost(g)
           for size in BANK for g in BANK[size])
print("  (h_manhattan_scaled stays optimal on every one of them — admissibility is the difference.)")

## Task 2 — the duel: LLM vs. A* (headless demo)

`duel.py` gives the exact prompt, a checker, and the four scoring categories —
`illegal` / `suboptimal` / `wrong_cost` / `correct`. The load-bearing rule: **never
trust the model's self-reported cost; always recompute with the checker** (that is why
`wrong_cost` exists as its own category).

We run headless with the deterministic `echo` backend so this notebook executes
anywhere.

> **`echo` is a deterministic FAKE, not a model.** It returns a fixed string, so
> `parse_reply` finds no `PATH:`/`COST:` and every reply scores `illegal` — that only
> exercises the *pipeline*, not a model. For a real scaling curve (A* flat at 100%, LLM
> declining), re-run with `backend="ollama"` or `backend="manual"`. Never report `echo`
> output as a model result.

In [ ]:
import os
from aicourse.llm import LLM
from duel import prompt_for, parse_reply, score_llm_reply

backend = os.environ.get("AICOURSE_NB_BACKEND", "echo")   # echo = headless-safe fake
llm = LLM(backend=backend)

grid = BANK[5][0]
print("Sample 5x5 grid:")
print(render(grid))
print("Ground-truth optimum via A*:", optimal_cost(grid), "\n")

# pipeline on a couple of instances (echo => all 'illegal', as noted above)
from collections import Counter
cats = Counter()
for g in BANK[5][:3]:
    reply = llm.complete(prompt_for(g)).text
    path, cost = parse_reply(reply)
    cats[score_llm_reply(g, path, cost).category] += 1
print("backend:", backend, "(echo = FAKE)" if backend == "echo" else "")
print("category counts (echo pipeline):", dict(cats))

# Prove the checker catches a self-reported-cost lie: hand it the true optimal PATH
# but a WRONG reported cost -> must be classified 'wrong_cost', not 'correct'.
node, _ = astar(GridProblem(grid), solution.h_manhattan_scaled)
optimal_path = "".join(node.path())
scored = score_llm_reply(grid, optimal_path, node.g + 999)
print(f"\noptimal path {optimal_path!r} with a LIED cost -> category={scored.category!r} "
      f"(checker_cost={scored.checker_cost}, optimal={scored.optimal_cost})")
assert scored.category == "wrong_cost"
print("The checker recomputed the cost and refused the model's self-report (never trust it).")